In [ ]:
print("hello")

In [ ]:
from huggingface_hub import login
login("<REDACTED>")  # あなたの読み取り権限付きトークン

MODEL_DIR = "ayarnte/Idea_Reward_Model"  # repo_id が正しいか再確認
from transformers import AutoTokenizer, AutoModelForSequenceClassification

auth = {"token": True}  # 直前の login を使う
tok = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, **auth)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, use_safetensors=True, **auth)

In [ ]:
from irm_iclr import IRMScorer

# ★ここだけ変更：ローカルフォルダ → HFのrepo_id
MODEL_DIR = "ayarnte/Idea_Reward_Model"

MAX_LEN = 512
STRIDE_RATIO = 0.5

scorer = IRMScorer(
    model_dir=MODEL_DIR,            # ← HF上のモデルを直接読む
    max_length=MAX_LEN,
    window_stride_ratio=STRIDE_RATIO,
)

title = "A differentiable bisociation module for cross-domain idea generation"
body  = """We propose a simple yet effective module that encourages cross-domain association 
during idea generation. The model learns to connect distant concepts by optimizing a 
rank-augmented reward signal derived from peer-review score predictors..."""

res = scorer.score_ideas([title], [body])  # [{'raw_score': ..., 'reward': ...}]
print(res[0])


In [ ]:
import pandas as pd

titles = [
    "Self-supervised curation of literature for proposal writing",
    "Neural reviewer: modeling rebuttal sensitivity in peer-review",
    "Spectral reward shaping for creative large language models",
]
bodies = [
    "We introduce an automatic pipeline that mines and clusters prior work to help formulate new proposals...",
    "We study how rebuttal content shifts reviewer scoring and propose a model that predicts post-rebuttal changes...",
    "We shape the reward landscape using spectral properties of idea graphs to promote non-trivial associations...",
]

outs = scorer.score_ideas(titles, bodies)

df = pd.DataFrame([{
    "title": t,
    "raw_score": o["raw_score"],
    "reward_0_1": o["reward"],
} for t, o in zip(titles, outs)])

df.sort_values("reward_0_1", ascending=False).reset_index(drop=True)


In [ ]:
from datasets import load_dataset

ds = load_dataset("ayarnte/iclr25")
df_fromInternet = ds["train"].to_pandas()

df_fromInternet.head()

In [ ]:
df_fromLocal = pd.read_parquet("data/iclr/iclr25v2.parquet")
df_fromLocal.head()

In [ ]:
import os, json
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

from irm_iclr import (
    DataConfig, make_dataset, IRMScorer, make_examples,
    z_transform_with_stats,           # ← 追加
    # compute_year_stats の定義は同ファイル内。importされていなければ from irm_iclr import compute_year_stats
)
from irm_iclr import compute_year_stats  # 念のため別行で

# === 設定 ===
model_dir = "ayarnte/Idea_Reward_Model"
data_path = "data/iclr/iclr25v2.parquet"
target_type = "year_z"   # or "raw"
accept_threshold = 6.0
add_year_tag = True

use_sliding = True
stride_ratio = 0.75
agg = "median"
use_reward = True
max_len = 512
bs = 64
eval_subset = "valid"   # "train" / "valid"

# === データ再構築（examplesベース） ===
dcfg = DataConfig(
    data_path=data_path,
    model_name="allenai/scibert_scivocab_uncased",
    max_length=max_len,
    seed=123,
    train_ratio=0.9,
    target_type=target_type,
    accept_threshold=accept_threshold,
    add_year_tag=add_year_tag,
)

examples_all = make_examples(dcfg)  # ここには text/score/year/accept が入る（targetはまだ）
n_train = int(len(examples_all) * dcfg.train_ratio)
train_ex  = examples_all[:n_train]
valid_ex  = examples_all[n_train:]

# 学習統計で z を作成（year_z のとき）
if target_type == "year_z":
    year_stats, default_stats = compute_year_stats(train_ex)  # 学習側統計
    train_ex = z_transform_with_stats(train_ex, year_stats, default_stats)
    valid_ex = z_transform_with_stats(valid_ex, year_stats, default_stats)
else:
    for ex in train_ex: ex["target"] = float(ex["score"])
    for ex in valid_ex: ex["target"] = float(ex["score"])

examples = valid_ex if eval_subset == "valid" else train_ex

texts       = [e["text"] for e in examples]
targets     = np.array([e["target"] for e in examples], dtype=float)   # ← もう存在する
scores_raw  = np.array([e["score"]  for e in examples], dtype=float)
years       = np.array([e.get("year", None) for e in examples])
accept_true = (scores_raw >= accept_threshold).astype(int)

# === スコアラ（HFからロード）
scorer = IRMScorer(
    model_dir=model_dir,
    max_length=max_len,
    window_stride_ratio=stride_ratio,
    agg=agg,
)

# === 推論（バッチ）
preds = []
rewards = []
with torch.no_grad():
    for i in tqdm(range(0, len(texts), bs)):
        out = scorer.score_texts(texts[i:i+bs])  # [{'raw_score':..., 'reward':...}]
        preds.extend([d["raw_score"] for d in out])
        rewards.extend([d["reward"] for d in out])

preds   = np.array(preds, dtype=float)
rewards = np.array(rewards, dtype=float)
y_score = rewards if use_reward else preds

print("推論完了:", len(y_score))


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score

# y_score = rewards か preds は既に定義済み
# accept_true = 0/1 のラベルも既にある

auc = roc_auc_score(accept_true, y_score)
print("AUC =", auc)

# しきい値 0.5 で採択/不採択二値化（reward を使う場合は 0.5 が自然）
pred_accept = (y_score >= 0.5).astype(int)

print("Accuracy =", accuracy_score(accept_true, pred_accept))
print("F1 =", f1_score(accept_true, pred_accept))
print("Precision =", precision_score(accept_true, pred_accept))
print("Recall =", recall_score(accept_true, pred_accept))

from sklearn.metrics import f1_score

best_f1 = -1
best_th = None

for th in np.linspace(0, 1, 200):
    pred = (y_score >= th).astype(int)
    f1 = f1_score(accept_true, pred)
    if f1 > best_f1:
        best_f1 = f1
        best_th = th

print("Best threshold =", best_th)
print("Best F1 =", best_f1)


pred_accept = (y_score >= best_th).astype(int)

print("Accuracy =", accuracy_score(accept_true, pred_accept))
print("F1 =", f1_score(accept_true, pred_accept))
print("Precision =", precision_score(accept_true, pred_accept))
print("Recall =", recall_score(accept_true, pred_accept))



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(accept_true, y_score)
roc_auc = auc(fpr, tpr)

precision, recall, _ = precision_recall_curve(accept_true, y_score)
pr_auc = average_precision_score(accept_true, y_score)

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
plt.plot([0,1],[0,1],'--')
plt.title("ROC Curve")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend()
plt.show()

plt.figure(figsize=(5,5))
plt.plot(recall, precision, label=f"PR-AUC={pr_auc:.3f}")
plt.title("PR Curve")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(6,6))
plt.scatter(targets, preds, s=6, alpha=0.4)
z = np.polyfit(targets, preds, 1)
xs = np.linspace(targets.min(), targets.max(), 100)
plt.plot(xs, z[0]*xs + z[1], '--')
plt.xlabel("target (year_z or raw)")
plt.ylabel("prediction (logit)")
plt.title("Pred vs Target")
plt.show()


In [ ]:
from scipy.stats import spearmanr
df = pd.DataFrame({"year": years, "pred": preds, "tgt": targets})
per = df.groupby("year").apply(lambda g: spearmanr(g["pred"], g["tgt"]).correlation)

plt.figure(figsize=(7,4))
plt.bar(per.index.astype(str), per.values)
plt.ylim(0, 1)
plt.ylabel("Spearman")
plt.title("Per-Year Spearman")
plt.show()

per


In [ ]:
def calibration_bins(y_true_bin, y_score, n_bins=10):
    qs = np.quantile(y_score, np.linspace(0,1,n_bins+1))
    bins = []
    for i in range(n_bins):
        lo, hi = qs[i], qs[i+1]
        sel = (y_score >= lo) & (y_score <= hi if i==n_bins-1 else y_score < hi)
        idx = np.where(sel)[0]
        if len(idx)==0:
            bins.append((lo, hi, 0, np.nan, (lo+hi)/2))
            continue
        bins.append((lo, hi, len(idx), np.mean(y_true_bin[idx]), np.mean(y_score[idx])))
    return bins

bins = calibration_bins(accept_true, y_score)
pred_mean = [b[4] for b in bins]
true_mean = [b[3] for b in bins]

plt.figure(figsize=(5,5))
plt.plot([0,1],[0,1],'--')
plt.plot(pred_mean, true_mean, marker='o')
plt.xlabel("Predicted")
plt.ylabel("Empirical Accept Rate")
plt.title("Reliability (Calibration)")
plt.show()

bins


In [ ]:
def confusion(th):
    pred = (y_score >= th).astype(int)
    TP = ((pred==1)&(accept_true==1)).sum()
    FP = ((pred==1)&(accept_true==0)).sum()
    TN = ((pred==0)&(accept_true==0)).sum()
    FN = ((pred==0)&(accept_true==1)).sum()
    P = TP / max(TP+FP,1e-9)
    R = TP / max(TP+FN,1e-9)
    F1 = 2*P*R/max(P+R,1e-9)
    return F1, th, TP, FP, TN, FN, P, R

ths = np.unique(y_score)
best = max(confusion(th) for th in ths)
best
